quantum-cortex — session B re read on a mounted checkpoint, with every answer kept per line (Rev66)

Settings: Accelerator = None (CPU is enough, minutes), Internet = On. Secrets: `GITHUB_TOKEN`, `QUANTUM_CORTEX_JOURNAL_KEY` (the same key that sealed the run's journal). Input: the fine tune notebook's version whose Output holds `runs-keep/journal-n1-<V>/` (`ckpt.pt`, `journal.jsonl`, the payload store). Set `V` below. The cell reopens the sealed journal from the disk in a new process and re runs session B on the checkpoint: the numbers must equal the run's committed session B file (checked and printed); the file adds `answers` (entity, schema, the attribute asked, the answer, its outcome, the own line's mark, the cited line's mark, whether the cited line is the own one and the highest marked one) and, from now on, the marks and the oracle. Writes `metrics/mqar/hm-lm-<run_id>-session-b-answers.json`. Download the log as a file before reading any number.


In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # the repository is private
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
# the runs directory lives OUTSIDE the clone (/kaggle/working/runs-keep) and is never deleted: a session
# interruption that re-runs the notebook finds the last checkpoint there and resumes instead of restarting
import os
os.makedirs('/kaggle/working/runs-keep', exist_ok=True)
if not os.path.islink('runs'): os.symlink('/kaggle/working/runs-keep', 'runs')
print('runs ->', os.path.realpath('runs'), '| kept from a previous session:', os.listdir('runs'))
!pip -q install jsonschema datasets cryptography


In [ ]:
# the key: from the Kaggle secret to the environment, never to a file
import os
from kaggle_secrets import UserSecretsClient
os.environ['QUANTUM_CORTEX_JOURNAL_KEY'] = UserSecretsClient().get_secret('QUANTUM_CORTEX_JOURNAL_KEY')
assert len(bytes.fromhex(os.environ['QUANTUM_CORTEX_JOURNAL_KEY'])) == 32, 'the key must be 32 bytes hex'
print('journal key: present (not printed)')
# the checkpoint relay (cortex_data/relay.py): with these two secrets the run pushes its directory to a private
# Kaggle dataset after every checkpoint and pulls it back before a resume, so a relaunch in a fresh container
# restarts from the last thousand steps; without them everything runs as before
try:
    os.environ['KAGGLE_USERNAME'] = UserSecretsClient().get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = UserSecretsClient().get_secret('KAGGLE_KEY')
    from cortex_data import relay
    print('relay:', 'on' if relay.available() else 'off (kaggle command missing)')
except Exception as e:
    print('relay: off (secrets KAGGLE_USERNAME / KAGGLE_KEY not attached)')


In [ ]:
V = 'v9-seed3'                              # the run to re read: v6, v7, v9, v9-seed2, v9-seed3
CONFIGS = {'v6': 'configs/kaggle_t4_journal_v6.json', 'v7': 'configs/kaggle_t4_journal_v7.json',
           'v8': 'configs/kaggle_t4_journal_v8.json', 'v9': 'configs/kaggle_t4_journal_v9.json',
           'v9-seed2': 'configs/kaggle_t4_journal_v9_seed2.json', 'v9-seed3': 'configs/kaggle_t4_journal_v9_seed3.json'}
CFG = CONFIGS[V]                           # a plain name: the shell magic below substitutes {CFG}, never {CONFIGS[V]}
POLICIES = ['plain', 'mark-veto', 'mark-veto+value']     # ADR-008 amendments of 2026-09-14 and 2026-09-18
import glob, os, shutil, json, torch
cands = sorted(glob.glob(f'/kaggle/input/**/journal-n1-{V.replace("v9-seed", "v9-seed")}/ckpt.pt', recursive=True)) or \
        sorted(glob.glob(f'/kaggle/input/**/journal-n1-{V}/ckpt.pt', recursive=True))
assert cands, f'no Input holds journal-n1-{V}/ckpt.pt: mount the fine tune notebook version of that run'
src_dir = os.path.dirname(cands[0]); dst = f'runs-keep/journal-n1-{V}'
os.makedirs(dst, exist_ok=True)
for f in os.listdir(src_dir):
    s, d = os.path.join(src_dir, f), os.path.join(dst, f)
    if not os.path.exists(d):
        (shutil.copytree if os.path.isdir(s) else shutil.copy)(s, d)
run_id = torch.load(os.path.join(dst, 'ckpt.pt'), map_location='cpu', weights_only=False)['run_id']
print('run', run_id, '| config', CFG, '| checkpoint', dst)
KEYS = ('hm_recall_on', 'hm_valid_citation_on', 'hm_invalid_citation_on', 'hm_false_abstention_on',
        'hm_negctrl_rate', 'hm_attr_exact_given_valid', 'hm_retrieval_hit', 'gate_conditions_without_skill_arm')
for pol in POLICIES:
    suffix = '' if pol == 'plain' else '-' + pol.replace('+', '-')
    out = f'metrics/mqar/hm-lm-{run_id}-session-b-answers{suffix}.json'
    !python -m cortex_c2b.hm_lm --session-b --policy {pol} --ckpt {dst}/ckpt.pt --journal {dst}/journal.jsonl --config {CFG} --out {out}
    r = json.load(open(out))
    print('POLICY', pol, '->', out, {k: r.get(k) for k in KEYS})
probe = f'metrics/mqar/retrieval-probe-{run_id}.json'
!python -m cortex_c2b.hm_lm --retrieval-probe --ckpt {dst}/ckpt.pt --journal {dst}/journal.jsonl --config {CFG} --out {probe}
print('PROBE ->', probe, json.load(open(probe))['reading'])
a = json.load(open(f'metrics/mqar/hm-lm-{run_id}-session-b-answers.json'))['answers']['on']
print('outcomes (plain):', {o: sum(1 for x in a if x['outcome'] == o) for o in ('strict', 'valid', 'invalid', 'abstain')})
